In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    # api_key="ai",
    # model="openai/gpt-oss-20b",
    # base_url="http://192.168.0.110:8000/v1",
    api_key="ai",
    # model="meta-llama/Llama-3.1-8B-Instruct",
    model="unsloth/gemma-3-27b-it-bnb-4bit",
    base_url="http://192.168.0.110:8001/v1",
    temperature=0,
    max_tokens=3000
)

In [2]:
from typing import TypedDict, List, Dict, Any, Annotated
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_core.messages import AnyMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_community.vectorstores import FAISS

# ==========================================
# 1. 툴(Tools) 정의 및 모델 바인딩
# ==========================================

@tool
def VectorDB(query: str) -> str:
    """Use this to search the military doctrine vectorstore. Input must be a concise semantic search query."""
    embedding = OpenAIEmbeddings()
    save_vector = "./north_korean_tactics_faiss_new"
    vectorstore = FAISS.load_local(save_vector, embedding, allow_dangerous_deserialization=True)
    retriever = vectorstore.as_retriever()
    print(f"[VectorDB 결과] '{query}'에 대한 군사 교리 내용 수집 완료.")
    docs = retriever.invoke(query)
    combined_text = "\n\n".join([doc.page_content for doc in docs])
    print("*" * 20 + "검색된 문서 결과" + "*" * 20)
    print(combined_text)
    print("*" * 55)
    return combined_text

@tool
def LLM(instruction: str) -> str:
    """Use this to reason, summarize, extract, or synthesize information. Input must be a clear instruction."""
    print(f"[LLM 추론 결과] 지시사항('{instruction}')에 대한 추론 및 요약 완료.")
    return model.invoke(instruction).content
    

# 사용할 툴 리스트 등록
tools = [VectorDB, LLM]

# 유저님의 모델에 툴을 바인딩합니다.
# model_with_tools는 이제 일반 텍스트 대신 tool_calls를 반환할 수 있게 됩니다.
model_with_tools = model.bind_tools(tools)


# ==========================================
# 2. 그래프 State 정의 (messages 추가)
# ==========================================

class TaskStepSpec(BaseModel):
    plan: str
    evidence_id: str
    tool: str
    tool_input: str

class ReWOOPlanSpec(BaseModel):
    steps: List[TaskStepSpec]

class ReWOOState(TypedDict):
    task: str
    steps: List[Dict[str, Any]]
    results: Dict[str, str]
    current_step_idx: int
    final_answer: str
    # ToolNode와 연동하기 위해 메시지 내역을 상태에 포함시킵니다.
    messages: Annotated[List[AnyMessage], add_messages]


# ==========================================
# 3. 보조 함수 (이전 증거 치환)
# ==========================================

def substitute_evidence(text: str, results: Dict[str, str]) -> str:
    for eq_id, val in results.items():
        text = text.replace(eq_id, str(val))
    return text


# ==========================================
# 4. LangGraph 노드(Nodes) 리팩토링
# ==========================================

planner_chain = ChatPromptTemplate.from_messages([
    ("system", """You are a planning module that breaks down a given task into sequential evidence collection and reasoning steps.
        Your task:
        Create a step-by-step plan to answer the given question. Each step must specify:
        - a detailed natural-language plan
        - one evidence variable ID
        - one external tool
        - one tool input
        
        Available tools:
        1. VectorDB
        Use this to search a vectorstore containing embedded military doctrines, field manuals, and related doctrinal references.
        Use VectorDB when the step requires doctrinal evidence, definitions, concepts, tactics, operational methods, terminology, or source-grounded military information.
        The tool_input must be a concise semantic search query.
        
        2. LLM
        Use this to reason over previous evidence, summarize retrieved evidence, extract definitions or lists, compare concepts, or synthesize the final answer.
        The tool_input must be a clear instruction. It may refer to previous evidence variables such as #E1 or #E2.
        
        Output requirements:
        - Return only the structured output object expected by the schema.
        - Do not include markdown.
        - Do not include explanations outside the object.
        - The top-level object must contain a field named steps.
        - steps must be a non-empty list.
        - Create enough steps to fully answer the task.
        - Each step must contain exactly these fields:
          - plan
          - evidence_id
          - tool
          - tool_input
        - evidence_id must start at #E1 and increment by 1 for each step: #E1, #E2, #E3, ...
        - Each step must have exactly one evidence_id.
        - tool must be exactly one of:
          - VectorDB
          - LLM
        - Do not use any other tool name.
        - Do not use Google, Search, Browser, Calculator, WolframAlpha, Python, DoctrineSearch, Vectorstore, or any other tool name.
        - plan must not be empty.
        - tool_input must not be empty.
        - Never return an empty steps list.
        
        Planning rules:
        - For military or doctrine-related tasks, use VectorDB before LLM.
        - Use VectorDB to gather source evidence.
        - Use LLM to extract, compare, reason, or synthesize from evidence.
        - The final step should usually be an LLM step that directly answers the task using the previous evidence.
        - If the task asks to compare two concepts, search for each concept separately, then use LLM to compare them.
        - If the task asks for a definition or list, one VectorDB search and one or two LLM extraction/final-answer steps are enough.
        - If the task has multiple dimensions, such as terrain, timing, enemy vulnerabilities, function, risk, coordination, or control requirements, search for the main dimensions separately when useful.
        - Keep the plan focused on the user question. Do not search for the opposite actor or reverse the direction of the question.
        - Do not assume the answer in the plan. Search for evidence first, then extract or synthesize from it.
        - Use broad semantic search queries rather than exact quoted phrases.
        - Do not put quotation marks around VectorDB search queries unless the exact phrase itself is essential.
        - Tool inputs for VectorDB should include the key military concept, actor, and requested comparison or dimension.
        - Tool inputs for LLM should explicitly say which evidence variables to use.
        
        Good VectorDB query style:
        - action unit doctrine function risk task organization planning
        - enabling unit doctrine support function mission allocation planning
        - North Korea terrain exploitation technological inferiority mountains tunnels concealment doctrine
        - North Korea adaptive operations disrupt enemy command control communications logistics infiltration
        - PMESII-PT operational variables doctrine
        - kill box joint fires airspace control fire support coordination dimensions
        
        Bad VectorDB query style:
        - "action unit" doctrine definition
        - EIW doctrine
        - tech-superior coalition C2 disruption tactics, when the question asks how North Korea disrupts enemy C2
        - hometown of #E2
        - using #E1 summarize the evidence
        
        Example task:
        Compare the roles of fixing drills and EIW in restricting enemy movement and influencing decisions.
        
        Example structured output:
        {{
          "steps": [
            {{
              "plan": "Search the doctrine vectorstore for passages defining fixing drills and explaining how they are used to fix, restrict, or shape enemy movement during operations.",
              "evidence_id": "#E1",
              "tool": "VectorDB",
              "tool_input": "fixing drills doctrine fix enemy restrict movement influence decision making"
            }},
            {{
              "plan": "Search the doctrine vectorstore for passages defining EIW and explaining how it affects enemy movement, decision-making, command and control, or operational behavior.",
              "evidence_id": "#E2",
              "tool": "VectorDB",
              "tool_input": "EIW doctrine definition restrict enemy movement influence decisions command control"
            }},
            {{
              "plan": "Compare the doctrinal roles of fixing drills and EIW using the retrieved evidence, focusing on how each restricts enemy movement and influences enemy decisions.",
              "evidence_id": "#E3",
              "tool": "LLM",
              "tool_input": "Using #E1 and #E2, compare fixing drills and EIW in how they restrict enemy movement and influence enemy decisions."
            }},
            {{
              "plan": "Produce a concise final answer that directly addresses the question and summarizes the key similarities and differences between fixing drills and EIW.",
              "evidence_id": "#E4",
              "tool": "LLM",
              "tool_input": "Using #E3, provide the final answer comparing the roles of fixing drills and EIW in restricting enemy movement and influencing decisions."
            }}
          ]
        }}
        
        Now create a structured plan for the task below.
        
        Task:
        {task}
        """ ),
    ("human", "Task: {task}")
]) | model.with_structured_output(ReWOOPlanSpec)

def plan_node(state: ReWOOState) -> dict:
    print("🤖 [Node: Plan] 계획 수립 중...")
    response = planner_chain.invoke({"task": state["task"]})
    #plan 내용 디버깅
    plans = [
        f"plan: {step.plan}\n{step.evidence_id} = {step.tool}[{step.tool_input}]\n\n"
        for step in response.steps
    ]
    print("\n".join(plans))
    ###
    steps_dict = [step.model_dump() for step in response.steps]
    return {"steps": steps_dict, "current_step_idx": 0, "results": {}, "messages": []}


def execute_node(state: ReWOOState) -> dict:
    """[변경] 직접 실행하지 않고, 바인딩된 모델에게 툴 호출(tool_calls)을 유도합니다."""
    idx = state["current_step_idx"]
    step = state["steps"][idx]
    
    # 예: #E1 결과를 뒤단계 쿼리에 주입
    resolved_input = substitute_evidence(step["tool_input"], state["results"])
    
    print(f"⚙️ [Node: Execute] 모델에게 {step['tool']} 호출 요청 중... ({step['evidence_id']})")
    
    # 모델에게 플래너가 지정한 툴과 입력값을 강제로 매칭하여 실행하도록 컨텍스트를 줍니다.
    # prompt = f"""You must execute the current plan step.
    # Plan Description: {step['plan']}
    # Required Tool: {step['tool']}
    # Argument/Input: {resolved_input}
    
    # Call the designated tool with the provided argument immediately."""

    prompt = f"""You must execute the current plan step.
    Plan Description: {step['plan']}
    Required Tool: {step['tool']}
    Argument/Input: {resolved_input}
    
    Call the designated tool with the provided argument immediately.
    
    To use a tool, you answer tool you MUST respond ONLY with a JSON object in the following format (do not add conversational text):
    ```json
    {{
        "name": "VectorDB",
        "args": {{ "query": 'North Korea nuclear deterrence doctrine threat Seoul" }}
    }}
    If you don't need to use a tool, answer directly in natural language.
    """

    # 툴이 바인딩된 모델을 호출하면 내부적으로 tool_calls가 담긴 AIMessage가 반환됩니다.
    ai_message = model_with_tools.invoke(prompt)

    print("*" * 55)
    print(f"\n 모델 tool call 유도 내용:{ai_message}")
    # 이 메시지를 리턴하면 상태의 messages에 추가되어 다음 노드인 ToolNode가 읽을 수 있게 됩니다.
    return {"messages": [ai_message]}


def post_execute_node(state: ReWOOState) -> dict:
    """[추가] ToolNode가 실행한 결과를 ReWOO의 변수(#E) 스토어에 매핑합니다."""
    idx = state["current_step_idx"]
    step = state["steps"][idx]
    evidence_id = step["evidence_id"]
    
    # ToolNode가 실행을 마치면 최신 메시지(messages[-1])에 ToolMessage가 들어옵니다.
    tool_message = state["messages"][-1]
    tool_result = tool_message.content
    print("*" * 55)
    print(f"\ntool message 내용:{tool_result}")
    
    print(f"✅ [Node: Post-Execute] {evidence_id} 결과 저장 완료.")
    
    updated_results = {**state["results"], evidence_id: tool_result}
    
    return {
        "results": updated_results,
        "current_step_idx": idx + 1 # 다음 단계 스텝으로 인덱스 전환
    }


def should_continue(state: ReWOOState) -> str:
    if state["current_step_idx"] < len(state["steps"]):
        return "continue"
    return "end"


def final_answer_node(state: ReWOOState) -> dict:
    print("📝 [Node: Final Answer] 최종 답변 정리 중...")
    last_evidence_id = f"#E{len(state['steps'])}"
    final_raw_result = state["results"].get(last_evidence_id, "답변 생성 실패")
    return {"final_answer": final_raw_result}


# ==========================================
# 5. 워크플로우 그래프 빌드 (ToolNode 배치)
# ==========================================

workflow = StateGraph(ReWOOState)

# 전역 ToolNode 선언 (생성해 둔 툴 리스트 주입)
standard_tool_node = ToolNode(tools)

# 노드 등록
workflow.add_node("planner", plan_node)
workflow.add_node("executor", execute_node)
workflow.add_node("tools", standard_tool_node) # 👈 랭그래프 Prebuilt 툴 노드
workflow.add_node("post_executor", post_execute_node)
workflow.add_node("final_compiler", final_answer_node)

# 에지 연결 흐름 변경
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "executor")
workflow.add_edge("executor", "tools")         # 1. 모델이 tool_call 뱉으면 -> 툴 노드로
workflow.add_edge("tools", "post_executor")   # 2. 툴 노드가 실행 완료하면 -> 사후 처리 노드로

# 루프 분기점 위치 변경 (사후 처리 노드 끝난 후 체크)
workflow.add_conditional_edges(
    "post_executor",
    should_continue,
    {
        "continue": "executor",
        "end": "final_compiler"
    }
)

workflow.add_edge("final_compiler", END)
rewoo_agent = workflow.compile()

C:\Users\user\AppData\Local\Temp\ipykernel_39912\3895464281.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
query = "How do action and enabling units differ in function, risk, and mission task allocation during planning?"

embedding = OpenAIEmbeddings()
save_vector = "./north_korean_tactics_faiss_new"
vectorstore = FAISS.load_local(save_vector, embedding, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever()
docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in docs])

# 2. 컨텍스트 기반 답변 생성 프롬프트
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert military analyst. Answer the user's question accurately using ONLY the provided context document pieces. If the context lacks information, do your best with given facts."),
    ("human", "Context:\n{context}\n\nQuestion: {query}")
])

naive_chain = prompt | model
response = naive_chain.invoke({"context": context, "query": query})
print(response.content)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [4]:
query = "Analyze how NK’s nuclear deterrence and forward-deployed forces mutually reinforce threats to Seoul."

# 그래프 실행 (초기 상태 주입)
initial_state = {"task": query}

# 징검다리처럼 상태가 업데이트되는 과정을 스트리밍으로 확인하기
for event in rewoo_agent.stream(initial_state):
    for node_name, state_update in event.items():
        print(f"\n[업데이트 됨] 노드: {node_name}")
        
        # plan 노드가 끝나면 'steps' 정보가 state에 가득 차게 됨
        if "steps" in state_update:
            print(f"👉 세워진 계획 개수: {len(state_update['steps'])}개")
            
        # execute 노드가 반복될 때마다 results 딕셔너리에 #E1, #E2가 누적 업데이트 됨
        if "results" in state_update:
            print(f"👉 현재 확보된 증거 변수들: {list(state_update['results'].keys())}")

        # 마지막 final_compiler 노드가 수행되면 그 결과물에서 최종 답변을 가로챕니다.
        if node_name == "final_compiler":
            final_answer = state_update.get("final_answer", "")

# 최종 결과물 출력
print("\n" + "="*40 + "\n🎯 최종 답변:\n" + "="*40)
print(final_answer)

# 최종 결과물만 깔끔하게 출력
# final_state = rewoo_agent.invoke(initial_state)
# print("\n" + "="*40 + "\n🎯 최종 답변:\n" + "="*40)
# print(final_state["final_answer"])

🤖 [Node: Plan] 계획 수립 중...
plan: Search the doctrine vectorstore for information on North Korea's nuclear deterrence strategy, including its stated goals, capabilities, and operational concepts related to Seoul.
#E1 = VectorDB[North Korea nuclear deterrence Seoul threats doctrine strategy capabilities]


plan: Search the doctrine vectorstore for information on North Korea’s forward-deployed conventional forces, their disposition near the DMZ, and their potential roles in an attack against Seoul.
#E2 = VectorDB[North Korea forward deployed forces DMZ Seoul attack conventional forces]


plan: Analyze the retrieved evidence to identify how North Korea's nuclear capabilities might enable or enhance the effectiveness of its forward-deployed forces in threatening Seoul.
#E3 = LLM[Using #E1 and #E2, analyze how North Korea’s nuclear deterrence and forward-deployed forces mutually reinforce threats to Seoul. Focus on how nuclear capabilities enable conventional force operations.]


plan: Analyz

In [23]:
import json

# 파일 경로 설정 (본인의 환경에 맞게 수정하세요)
input_file_path = "golden_answers_final_master_file.json"       # 원본 JSON 파일
output_file_path = "question.json"  # 저장할 새 JSON 파일

try:
    # 1. 원본 JSON 파일 불러오기
    with open(input_file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # 만약 JSON 데이터가 {"steps": [...]} 구조라면 내부 리스트만 쏙 빼옵니다.
    # 만약 처음부터 리스트 형태([...])로 저장되어 있었다면 data를 그대로 씁니다.
    steps = data.get("steps", data) if isinstance(data, dict) else data
    
    # 2. 리스트 컴프리헨션으로 'tool_input' 값만 뽑아서 묶기
    # 💡 만약 JSON 안의 키 이름이 진짜 'input' 이라면 아래 "tool_input"을 모두 "input"으로 고치시면 됩니다!
    extracted_inputs = [
        step["input"] for step in steps if "input" in step
    ]
        
    # 3. 새로운 JSON 파일로 저장하기
    with open(output_file_path, "w", encoding="utf-8") as f:
        # indent=4 : 파일 안의 데이터를 사람이 보기 좋게 들여쓰기 해줍니다.
        # ensure_ascii=False : 군사 교리 같은 텍스트에 한글이나 특수문자가 섞여있을 때 깨지지 않게 보존합니다.
        json.dump(extracted_inputs, f, ensure_ascii=False, indent=4)
        
    print(f"✅ 추출 완료! 새 파일이 생성되었습니다 ➡️ {output_file_path}")
    print("✨ 추출된 값 목록:")
    print(extracted_inputs)

except FileNotFoundError:
    print(f"❌ {input_file_path} 파일을 찾을 수 없습니다. 파일명을 확인해 주세요.")
except Exception as e:
    print(f"❌ 에러 발생: {e}")

✅ 추출 완료! 새 파일이 생성되었습니다 ➡️ question.json
✨ 추출된 값 목록:
['Analyze how NK’s nuclear deterrence and forward-deployed forces mutually reinforce threats to Seoul.', 'If NK faces a tech-superior coalition, how might adaptive ops exploit terrain and disrupt enemy C2?', 'How does NK doctrine employ deception, echeloning, and terrain in both offensive and defensive ops vs. tech-superior foes?', 'Describe how North Korea mitigates tech inferiority by exploiting terrain, timing, and enemy vulnerabilities.', 'Which eight operational variables—PMESII-PT—define an OE for military analysis and planning?', 'Analyze how KPA adapts/evolves across OE variables to counter U.S. power, considering threat motivations.', 'If enabling units faced unexpected heavy losses, how might this affect the action unit’s mission outcome?', 'How do action and enabling units differ in function, risk, and mission task allocation during planning?', 'How do action and enabling forces differ in organization and mission roles with

In [42]:
import json
from typing import List, Dict, Any
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


# ==========================================
# 1. 평가 타겟 질문 리스트 (앞서 선정한 10개)
# ==========================================
eval_questions = [
    "Analyze how NK’s nuclear deterrence and forward-deployed forces mutually reinforce threats to Seoul.",
    # "How does NK doctrine employ deception, echeloning, and terrain in both offensive and defensive ops vs. tech-superior foes?",
    # "Describe how North Korea mitigates tech inferiority by exploiting terrain, timing, and enemy vulnerabilities.",
    # "How do action and enabling units differ in function, risk, and mission task allocation during planning?",
    # "Compare kill zone vs. kill box in KPAGF ops, including dimensionality and coordination/control requirements.",
    # "Compare personnel and major equipment allocations among KPAGF armored, mechanized, and infantry divisions.",
    # "How do KPA air assault defense reserves use AAAD, kill zones, and MANPADS to counter enemy landings?",
    # "Compare KPA active vs. passive air defense measures in countering enemy air superiority and detection.",
    # "Compare how integrated vs. dispersed attacks differ in objectives, target selection, and force employment.",
    # "Analyze how KPA’s info attacks, data manipulation, and perception management interconnect to disrupt adversaries."
]

embedding = OpenAIEmbeddings()
save_vector = "./north_korean_tactics_faiss_new"
vectorstore = FAISS.load_local(save_vector, embedding, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever()


# ==========================================
# 2. LLM as a Judge 구조화 출력 스키마 정의
# ==========================================
class JudgeEvaluationSpec(BaseModel):
    naive_score: int = Field(description="Score for Naive RAG response (0-10)", ge=0, le=10)
    rewoo_score: int = Field(description="Score for ReWOO Agent response (0-10)", ge=0, le=10)
    dimension_coverage_analysis: str = Field(
        description="Analyze how well each system covered all requested dimensions (e.g., comparing multiple concepts or handling multiple sub-tactics). Write within 100 characters"
    )
    quality_shift_analysis: str = Field(
        description="Detailed analysis focusing specifically on the QUALITY SHIFT (improvement, depth, logical synthesis) from Naive RAG to ReWOO. Write within 100 characters"
    )
    verdict: str = Field(
        description="Final verdict on whether the ReWOO system successfully solved the keyword dilution/fragmentation problem of Naive RAG. Write within 100 characters"
    )

# ==========================================
# 3. 기본 Retriever (Naive RAG) 파이프라인 함수
# ==========================================
def run_naive_rag(query: str, retriever: Any, model: Any) -> str:
    """단순 쿼리 투입 후 관련 문서를 통째로 긁어와 답변을 생성하는 기본 RAG 방식"""
    # 1. 질문 그대로 Retriever에 투입
    docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # 2. 컨텍스트 기반 답변 생성 프롬프트
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert military analyst. Answer the user's question accurately using ONLY the provided context document pieces. If the context lacks information, do your best with given facts."),
        ("human", "Context:\n{context}\n\nQuestion: {query}")
    ])
    
    naive_chain = prompt | model
    response = naive_chain.invoke({"context": context, "query": query})
    return response.content

# ==========================================
# 4. LLM Judge 노드 및 실행 루프 정의
# ==========================================
# 유저님의 모델 인스턴스 설정 (예: judge용은 능력이 뛰어난 gpt-4o 등을 추천)


# 판사 LLM에게 내릴 엄격한 평가 프롬프트
judge_system_prompt = """You are a rigorous Quality Assurance Judge evaluating two advanced Military RAG architectures.
Your core mission is to analyze the QUALITY SHIFT between a 'Naive RAG' system and a 'ReWOO Agent' system.

Evaluation Focus:
1. Multi-dimensionality: Complex queries ask for multiple dimensions (e.g., Active vs Passive, Terrain vs Timing vs Vulnerability). Did Naive RAG suffer from 'Keyword Dilution' (focusing only on one heavily-weighted keyword and missing others)?
2. Synthesis Depth: Did the system just list facts (dictionary style), or did it logically interconnect the concepts as requested by the query?
3. Completeness: Did the system miss any critical comparison points?

Compare the two responses critically and provide scores out of 10 along with a detailed breakdown of the quality delta.

Write each analysis field very simply, summarizing only the key points in 2 to 3 sentences (maximum 200 characters).
"""

judge_prompt_template = ChatPromptTemplate.from_messages([
    ("system", judge_system_prompt),
    ("human", """[Target Question]
{query}

[System A: Naive RAG Response]
{naive_response}

[System B: ReWOO Agent Response]
{rewoo_response}

Evaluate the quality shift from Naive RAG to ReWOO based on the criteria. Provide the output strictly conforming to the requested schema.""")
])

# 구조화된 출력을 보장하는 판사 체인 생성
judge_chain = judge_prompt_template | model.with_structured_output(JudgeEvaluationSpec)


# ==========================================
# 5. 전체 벤치마크 런타임 실행 및 결과 저장
# ==========================================
def run_evaluation_benchmark(rewoo_agent, retriever, base_model):
    evaluation_report = []
    
    print("🚀 [Benchmark] ReWOO vs Naive RAG 평가 시스템을 시작합니다.")
    
    for idx, query in enumerate(eval_questions):
        print(f"\n🎯 [{idx+1}/{len(eval_questions)}] 질문 분석 중: {query}")
        
        try:
            # 1. Naive RAG 답변 생성
            print("  - Naive RAG 구동 중...")
            naive_resp = run_naive_rag(query, retriever, base_model)
            
            # 2. ReWOO Agent 답변 생성
            print("  - ReWOO Agent 구동 중...")
            rewoo_output = rewoo_agent.invoke({"task": query})
            rewoo_resp = rewoo_output.get("final_answer", "답변 생성 실패")
            
            # 3. LLM 판사를 통한 교차 검증 및 점수 산정
            print("  - ⚖️ LLM 판사 평가 진행 중...")
            evaluation: JudgeEvaluationSpec = judge_chain.invoke({
                "query": query,
                "naive_response": naive_resp,
                "rewoo_response": rewoo_resp
            })
            
            # 4. 결과 누적
            result_item = {
                "id": idx + 1,
                "question": query,
                "naive_score": evaluation.naive_score,
                "rewoo_score": evaluation.rewoo_score,
                "score_delta": evaluation.rewoo_score - evaluation.naive_score,
                "dimension_analysis": evaluation.dimension_coverage_analysis,
                "quality_shift_analysis": evaluation.quality_shift_analysis,
                "verdict": evaluation.verdict,
                "raw_responses": {
                    "naive": naive_resp,
                    "rewoo": rewoo_resp
                }
            }
            evaluation_report.append(result_item)
            print(f"  - 📊 점수 결과 -> Naive: {evaluation.naive_score} | ReWOO: {evaluation.rewoo_score} (Delta: {result_item['score_delta']})")
            
        except Exception as e:
            print(f"  - ❌ 질문 처리 중 에러 발생: {e}")
            
    # 5. 리포트를 JSON 파일로 깔끔하게 저장
    output_filename = "rewoo_benchmark_report.json"
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(evaluation_report, f, ensure_ascii=False, indent=4)
        
    print(f"\n🏁 [Benchmark 완료] 모든 평가가 완료되었습니다! 결과 저장 완료 ➡️ {output_filename}")

# 실행 예시 (주석을 해제하고 변수를 매핑하여 사용하세요)
run_evaluation_benchmark(rewoo_agent=rewoo_agent, retriever=retriever, base_model=model)

🚀 [Benchmark] ReWOO vs Naive RAG 평가 시스템을 시작합니다.

🎯 [1/10] 질문 분석 중: Analyze how NK’s nuclear deterrence and forward-deployed forces mutually reinforce threats to Seoul.
  - Naive RAG 구동 중...
  - ReWOO Agent 구동 중...
🤖 [Node: Plan] 계획 수립 중...
plan: Use the doctrine vectorstore to locate passages that define North Korea’s nuclear deterrence strategy and explain how it positions and threatens Seoul as a strategic target.
#E1 = VectorDB[North Korea nuclear deterrence doctrine threat to Seoul]


plan: Search the doctrine vectorstore for information on North Korea’s forward-deployed forces, their locations relative to Seoul, and doctrines that describe how these troops and assets enhance the threat posture against the city.
#E2 = VectorDB[North Korea forward-deployed forces near Seoul threat doctrine]


plan: Extract the key points from the evidence in #E1 and #E2 that describe the individual threat mechanisms of nuclear deterrence and forward-deployed forces, noting how each is intended to af